In [1]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

Phase 1

In [3]:
movies = pd.read_csv("C:\\Users\\sarum\\OneDrive\\Desktop\\Data AI Training\\Movie Recommendation\\tmdb_5000_movies.csv")
credits = pd.read_csv("C:\\Users\\sarum\\OneDrive\\Desktop\\Data AI Training\\Movie Recommendation\\tmdb_5000_credits.csv")

In [4]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [5]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [6]:
credits["crew"][0] 

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [7]:
movies = movies.merge(credits, on="title")

In [8]:
movies = movies[['movie_id','title','overview',
                 'genres','keywords','cast','crew']]

In [9]:
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

Phase2

In [10]:
def convert(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [11]:
# extracting director name from crew column
def convert_cast(obj):
    return [i['name'] for i in ast.literal_eval(obj)[:3]]
# lead actors can influence movie so we are selecting top 3
movies['cast'] = movies['cast'].apply(convert_cast)

In [12]:
def get_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            return i['name']
    return ""

movies['crew'] = movies['crew'].apply(get_director)


In [13]:
# split overview into list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# remove spaces from strings in genres, keywords, cast and crew columns
for col in ['genres','keywords','cast']:
    movies[col] = movies[col].apply(
        lambda x: [i.replace(" ","") for i in x]
    )

# remove spaces from director name in crew column
movies['crew'] = movies['crew'].apply(
    lambda x: x.replace(" ","")
)
#need to look

In [14]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew'].apply(lambda x: [x])
)

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))


In [15]:
movies.columns

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags'],
      dtype='str')

In [16]:
movies["tags"][5]

"The seemingly invincible Spider-Man goes up against an all-new crop of villain – including the shape-shifting Sandman. While Spider-Man’s superpowers are altered by an alien organism, his alter ego, Peter Parker, deals with nemesis Eddie Brock and also gets caught up in a love triangle. Fantasy Action Adventure dualidentity amnesia sandstorm loveofone'slife forgiveness spider wretch deathofafriend egomania sand narcism hostility marvelcomic sequel superhero revenge TobeyMaguire KirstenDunst JamesFranco SamRaimi"

Phase3

In [17]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(movies['tags']).toarray()


In [18]:
#[0.12, 0.00, 0.45, 0.00, ..., 0.02]

In [19]:
similarity = cosine_similarity(vectors)

Phase 4

In [20]:

pickle.dump(movies.reset_index(drop=True), open("movies.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))

In [21]:
def recommend(movie):
    index = movies[movies['title'] == movie].index[0]
    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f"\nRecommended movies similar to '{movie}':\n")
    for i in movie_list:
        print(movies.iloc[i[0]].title)


In [22]:
recommend("Avatar")


Recommended movies similar to 'Avatar':

Aliens
Falcon Rising
Battle: Los Angeles
Apollo 18
Star Trek Into Darkness


In [23]:
movies = pickle.load(open('movies.pkl', 'rb'))
similarity = pickle.load(open('similarity.pkl', 'rb'))

In [24]:
def recommend(movie_name, top_n=5):

    movie_name = movie_name.lower().strip()
    titles = movies['title'].str.lower().values

    if movie_name not in titles:
        return ["Movie not found in database"]

    index = movies[movies['title'].str.lower() == movie_name].index[0]

    # Safety check
    if index >= similarity.shape[0]:
        return ["Internal error: similarity matrix mismatch"]

    distances = list(enumerate(similarity[index]))
    distances = sorted(distances, key=lambda x: x[1], reverse=True)

    recommendations = []
    for i in distances[1:top_n+1]:
        recommendations.append(movies.iloc[i[0]].title)

    return recommendations


In [25]:
print(recommend("Avengers"))

['Movie not found in database']


In [26]:
def recommend_smart(movie_name, top_n=5):

    movie_name = movie_name.lower().strip()
    titles = movies['title'].str.lower()

    # Exact match check
    if movie_name in titles.values:
        idx = titles[titles == movie_name].index[0]

    else:
        # Partial match search (e.g., "harry", "avatar")
        matches = titles[titles.str.contains(movie_name)]
        if matches.empty:
            return {
                "error": "Movie not found",
                "suggestions": []
            }
        idx = matches.index[0]  # use the first partial match for recommendations

    # Get similarity scores and sort them
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    return [movies.iloc[i[0]].title for i in sim_scores]

In [27]:
print(recommend_smart("Avngers", 5)) # wrong spelling, should still work

{'error': 'Movie not found', 'suggestions': []}


In [28]:
print(recommend_smart("Avengers", 5)) # right spelling, should work as well

['Iron Man', 'Iron Man 3', 'The Avengers', 'Iron Man 2', 'Captain America: Civil War']


In [30]:
# Smart movie recommend function based on user input
def recommend_movie():
    movie_name = input("Enter a movie name: ").lower().strip()
    titles = movies['title'].str.lower()

    # 1. Exact match
    if movie_name in titles.values:
        idx = titles[titles == movie_name].index[0]

    else:
        # 2. Partial matching
        matches = titles[titles.str.contains(movie_name, na=False)]

        if matches.empty:
            print("\n Movie not found. Please try another name.")
            return

        # Use the closest partial match
        idx = matches.index[0]
        print(f"\nShowing results for: {movies.iloc[idx].title}")

    # 3. Calculate similarity
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

    print("\nRecommended Movies:\n")
    for i in sim_scores:
        print("*", movies.iloc[i[0]].title)

recommend_movie()



Showing results for: Harry Potter and the Half-Blood Prince

Recommended Movies:

* Harry Potter and the Prisoner of Azkaban
* Harry Potter and the Order of the Phoenix
* Harry Potter and the Chamber of Secrets
* Harry Potter and the Goblet of Fire
* Harry Potter and the Philosopher's Stone
